## RAG APPLICATION USING TYPESENSE

In [1]:
import typesense

In [2]:
import os

#creating a collection using the API key from .env
client = typesense.Client({
      'nodes': [{
    'host': 'tobd21ghuvcmr46fp-1.a2.typesense.net',  # For Typesense Cloud use xxx.a1.typesense.net #can be local
    'port': '443',       # For Typesense Cloud use 443  for local 8080
    'protocol': 'https'    # For Typesense Cloud use https 
  }],
  'api_key': os.getenv('TYPESENSE_API_KEY'),
  'connection_timeout_seconds': 2
})

#we need to create a schema 
books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}

print(client.collections.create(books_schema))

{'created_at': 1773366794, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [3]:
client

In [4]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [5]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [20]:
#### Refactored: use shared RAG pipeline with Typesense backend
from src.config import AppConfig
from src.ingestion import load_text_and_pdfs, split_documents
from src.pipeline import rag_simple, build_groq_llm
from src.retrieval import TypesenseRAGRetriever
from src.vectorstores import TypesenseVectorStore

In [21]:
cfg = AppConfig()

# Build Typesense-backed vector store from the same project data
raw_docs = load_text_and_pdfs()
chunks = split_documents(raw_docs)

store = TypesenseVectorStore(collection_name=cfg.typesense.collection_name)
store.build_from_documents(chunks)
retriever = TypesenseRAGRetriever(store)
llm = build_groq_llm()

rag_simple("What skills is the interviewer looking for?", retriever, llm)

In [30]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)
# embeddings = HuggingFaceEmbedding()

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3739.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:

docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": "tobd21ghuvcmr46fp-1.a2.typesense.net",  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key": os.getenv("TYPESENSE_API_KEY"),
    },typesense_collection_name="quality docu")


In [32]:
query = "most recognised frameworks specifically centred on timeliness/freshnes"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Yes â€” but an important point first: there are very few frameworks designed only for data timeliness in the data quality literature. Most frameworks treat timeliness as one dimension among several. However, there are specialised frameworks and models that focus primarily on the timeliness or freshness of information.

Below are the most recognised frameworks specifically centred on timeliness/freshness.


In [33]:
###retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x0000020FC41515B0>, search_kwargs={})

In [ ]:
query = 'most recognised frameworks specifically centred on timeliness/freshnes'
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content='Yes â€” but an important point first: there are very few frameworks designed only for data timeliness in the data quality literature. Most frameworks treat timeliness as one dimension among several. However, there are specialised frameworks and models that focus primarily on the timeliness or freshness of information.\n\nBelow are the most recognised frameworks specifically centred on timeliness/freshness.')

: 